In [1]:
# {1} Check Colab GPU – should show NVDIA A100 or T4 or similar
!nvidia-smi

Sun May 17 23:20:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             56W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# {2} Import torch and confirm CUDA
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

CUDA available: True
Device name: NVIDIA A100-SXM4-40GB


In [3]:
# {3} Set device for later training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using:', device)

Using: cuda


In [4]:
# {4} Install lightweight web deps (quiet)
!pip install flask pyngrok -q

In [5]:
# {5} Install Java tools (Colab already has OpenJDK)
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [6]:
# {6} Create project folders
!mkdir -p /content/ml-sprint/{data,models,java,sql}

In [7]:
# {7} Move to working dir
import os
os.chdir('/content/ml-sprint')
!pwd

/content/ml-sprint


In [8]:
# {8} Install Python ML stack
!pip install scikit-learn pandas joblib -q

In [9]:
# {9} Import core libs
import pandas as pd, numpy as np, sqlite3, joblib, json
from datetime import datetime

In [10]:
# {10} Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

In [11]:
# {11} Create SQLite database file
conn = sqlite3.connect('ml.db')
cur = conn.cursor()

In [12]:
# {12} Drop old tables if rerunning
cur.executescript('''
DROP TABLE IF EXISTS iris;
DROP TABLE IF EXISTS houses;
DROP TABLE IF EXISTS predictions;
DROP TABLE IF EXISTS api_logs;
DROP TABLE IF EXISTS metrics;
''')

In [13]:
# {13} Create iris table with engineered feature
cur.execute('''CREATE TABLE iris (id INTEGER PRIMARY KEY, sep_len REAL, sep_wid REAL, pet_len REAL, pet_wid REAL, class TEXT, pet_area REAL)''')

In [14]:
# {14} Create houses table
cur.execute('CREATE TABLE houses (id INTEGER PRIMARY KEY, sqft INT, price INT)')

In [15]:
# {15} Create predictions log
cur.execute('CREATE TABLE predictions (id INTEGER PRIMARY KEY, features TEXT, pred INT, ts TEXT)')

In [16]:
# {16} Create api_logs
cur.execute('CREATE TABLE api_logs (id INTEGER PRIMARY KEY, request TEXT, response TEXT, ts TEXT)')

In [17]:
# {17} Create metrics table
cur.execute('CREATE TABLE metrics (run TEXT, acc REAL)')

In [18]:
# {18} Commit schema
conn.commit()

In [19]:
# {19} Verify tables
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cur.fetchall())

[('iris',), ('houses',), ('predictions',), ('api_logs',), ('metrics',)]


In [20]:
# {20} Close for now
conn.close()

In [21]:
# {21} Load Iris from sklearn
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
df = iris.frame
df.columns = ['sep_len','sep_wid','pet_len','pet_wid','target']

In [22]:
# {22} Map target to names
names = iris.target_names
df['class'] = df['target'].apply(lambda x: names[x])

In [23]:
# {23} Add engineered feature pet_area
df['pet_area'] = df['pet_len'] * df['pet_wid']
df.head()

,sep_len,sep_wid,pet_len,pet_wid,target,class,pet_area
0,5.1,3.5,1.4,0.2,0,setosa,0.28
1,4.9,3.0,1.4,0.2,0,setosa,0.28
2,4.7,3.2,1.3,0.2,0,setosa,0.26
3,4.6,3.1,1.5,0.2,0,setosa,0.30
4,5.0,3.6,1.4,0.2,0,setosa,0.28


In [24]:
# {24} Save CSV for reference
df.to_csv('data/iris.csv', index=False)

In [25]:
# {25} Insert iris into SQLite
conn = sqlite3.connect('ml.db')
df[['sep_len','sep_wid','pet_len','pet_wid','class','pet_area']].to_sql('iris', conn, if_exists='append', index=False)

150

In [26]:
# {26} Insert sample houses
cur = conn.cursor()
houses = [(1,1000,200000),(2,1500,300000),(3,1200,240000),(4,1800,350000),(5,900,180000)]
cur.executemany('INSERT INTO houses VALUES (?,?,?)', houses)

In [27]:
# {27} Commit data
conn.commit()

In [28]:
# {28} Quick check iris count
pd.read_sql('SELECT COUNT(*) as n FROM iris', conn)

,n
0,150


In [29]:
# {29} Check houses
pd.read_sql('SELECT * FROM houses', conn)

,id,sqft,price
0,1,1000,200000
1,2,1500,300000
2,3,1200,240000
3,4,1800,350000
4,5,900,180000


In [30]:
# {30} Close connection
conn.close()

In [31]:
# {31} Prepare tensors for GPU training
X = df[['sep_len','sep_wid','pet_len','pet_wid']].values.astype('float32')
y = df['target'].values.astype('int64')

In [32]:
# {32} Standardize features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [33]:
# {33} Convert to torch tensors on GPU
X_t = torch.tensor(X, device=device)
y_t = torch.tensor(y, device=device)

In [34]:
# {34} Define simple logistic regression model
import torch.nn as nn
model = nn.Sequential(nn.Linear(4,3)).to(device)

In [35]:
# {35} Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [36]:
# {36} Training loop – runs on GPU
for epoch in range(200):
    optimizer.zero_grad()
    out = model(X_t)
    loss = criterion(out, y_t)
    loss.backward()
    optimizer.step()
    if (epoch+1)%50==0:
        print(f'Epoch {epoch+1}, loss {loss.item():.4f}')

Epoch 50, loss 0.4576
Epoch 100, loss 0.3520
Epoch 150, loss 0.2912
Epoch 200, loss 0.2463


In [37]:
# {37} Evaluate accuracy on same data
with torch.no_grad():
    pred = model(X_t).argmax(1)
    acc = (pred==y_t).float().mean().item()
print('Train accuracy:', acc)

Train accuracy: 0.9533333778381348


In [38]:
# {38} Save scaler and torch model
import pickle
with open('models/scaler.pkl','wb') as f: pickle.dump(scaler,f)
torch.save(model.state_dict(), 'models/iris_torch.pt')

In [39]:
# {39} Also save sklearn pipeline for Flask compatibility
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
pipe = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(max_iter=200))])
pipe.fit(df[['sep_len','sep_wid','pet_len','pet_wid']], y)
joblib.dump(pipe, 'models/iris_pipe.pkl')

['models/iris_pipe.pkl']

In [40]:
# {40} Log metric to SQLite
conn = sqlite3.connect('ml.db')
conn.execute('INSERT INTO metrics VALUES (?,?)', ('torch_gpu', acc))
conn.commit(); conn.close()

In [41]:
# {41} Show metrics table
pd.read_sql('SELECT * FROM metrics', sqlite3.connect('ml.db'))

,run,acc
0,torch_gpu,0.953333


In [42]:
# {42} Free GPU memory
torch.cuda.empty_cache()

In [43]:
# {43} Define predict function using saved pipe
import pandas as pd
pipe = joblib.load('models/iris_pipe.pkl')
def predict_iris(features):
    # Wrap features in a DataFrame to match training data format and avoid warnings
    df_features = pd.DataFrame([features], columns=['sep_len','sep_wid','pet_len','pet_wid'])
    return int(pipe.predict(df_features)[0])

In [44]:
# {44} Test predict
predict_iris([5.1,3.5,1.4,0.2])

0

In [45]:
# {45} Map to class name
class_name = ['setosa', 'versicolor', 'virginica'][predict_iris([5.1, 3.5, 1.4, 0.2])]
print(f"Predicted class: {class_name}")

Predicted class: setosa


In [46]:
# {46} Write Flask app to file
flask_code = '''
from flask import Flask, request, jsonify
import joblib, sqlite3, json
from datetime import datetime
import os
BASE=os.path.dirname(__file__)
model=joblib.load(os.path.join(BASE,'models/iris_pipe.pkl'))
DB=os.path.join(BASE,'ml.db')
app=Flask(__name__)
@app.post('/predict')
def predict():
    data=request.get_json()
    feats=data.get('features')
    pred=int(model.predict([feats])[0])
    con=sqlite3.connect(DB)
    con.execute('INSERT INTO predictions(features,pred,ts) VALUES(?,?,?)',(json.dumps(feats),pred,datetime.utcnow().isoformat()))
    con.execute('INSERT INTO api_logs(request,response,ts) VALUES(?,?,?)',(json.dumps(data),json.dumps({'pred':pred}),datetime.utcnow().isoformat()))
    con.commit(); con.close()
    return jsonify({'pred':pred,'class':['setosa','versicolor','virginica'][pred]})
@app.get('/')
def home(): return {'status':'ok'}
if __name__=='__main__': app.run(host='0.0.0.0',port=5000)
'''
with open('app.py','w') as f: f.write(flask_code)

In [47]:
# {47} Verify app.py created
!head -20 app.py


from flask import Flask, request, jsonify
import joblib, sqlite3, json
from datetime import datetime
import os
BASE=os.path.dirname(__file__)
model=joblib.load(os.path.join(BASE,'models/iris_pipe.pkl'))
DB=os.path.join(BASE,'ml.db')
app=Flask(__name__)
@app.post('/predict')
def predict():
    data=request.get_json()
    feats=data.get('features')
    pred=int(model.predict([feats])[0])
    con=sqlite3.connect(DB)
    con.execute('INSERT INTO predictions(features,pred,ts) VALUES(?,?,?)',(json.dumps(feats),pred,datetime.utcnow().isoformat()))
    con.execute('INSERT INTO api_logs(request,response,ts) VALUES(?,?,?)',(json.dumps(data),json.dumps({'pred':pred}),datetime.utcnow().isoformat()))
    con.commit(); con.close()
    return jsonify({'pred':pred,'class':['setosa','versicolor','virginica'][pred]})
@app.get('/')


In [48]:
# {48} Run Flask in background thread
from threading import Thread
import subprocess, time
def run_flask():
    subprocess.Popen(['python','app.py'])
Thread(target=run_flask, daemon=True).start()
time.sleep(3)
print('Flask started')

Flask started


In [49]:
# {49} Test locally with requests
!pip install requests -q
import requests
r = requests.post('http://127.0.0.1:5000/predict', json={'features':[5.1,3.5,1.4,0.2]})
print(r.json())

{'class': 'setosa', 'pred': 0}


In [50]:
# {50} Check prediction logged
pd.read_sql('SELECT * FROM predictions ORDER BY id DESC LIMIT 3', sqlite3.connect('ml.db'))

,id,features,pred,ts
0,1,"[5.1, 3.5, 1.4, 0.2]",0,2026-05-17T23:20:27.055238


In [51]:
# {51} Check api_logs
pd.read_sql('SELECT * FROM api_logs ORDER BY id DESC LIMIT 3', sqlite3.connect('ml.db'))

,id,request,response,ts
0,1,"{""features"": [5.1, 3.5, 1.4, 0.2]}","{""pred"": 0}",2026-05-17T23:20:27.055657


In [52]:
# {55} Flask still running in background

In [53]:
# {56} Check Java version again
!java -version

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [54]:
# {57} Download Weka stable jar (~10MB)
!wget -q https://repo1.maven.org/maven2/nz/ac/waikato/cms/weka/weka-stable/3.8.6/weka-stable-3.8.6.jar -O java/weka.jar

In [55]:
# {58} Verify jar
!ls -lh java/weka.jar

-rw-r--r-- 1 root root 9.4M Jan 27  2022 java/weka.jar


In [56]:
# {59} Create ARFF file for Weka
arff = '''@RELATION iris
@ATTRIBUTE sep_len NUMERIC
@ATTRIBUTE sep_wid NUMERIC
@ATTRIBUTE pet_len NUMERIC
@ATTRIBUTE pet_wid NUMERIC
@ATTRIBUTE class {setosa,versicolor,virginica}
@DATA
'''
with open('data/iris.arff','w') as f:
    f.write(arff)
    df[['sep_len','sep_wid','pet_len','pet_wid','class']].to_csv(f, header=False, index=False)

In [57]:
# {60} Preview ARFF
!head -n 15 data/iris.arff

@RELATION iris
@ATTRIBUTE sep_len NUMERIC
@ATTRIBUTE sep_wid NUMERIC
@ATTRIBUTE pet_len NUMERIC
@ATTRIBUTE pet_wid NUMERIC
@ATTRIBUTE class {setosa,versicolor,virginica}
@DATA
5.1,3.5,1.4,0.2,setosa
4.9,3.0,1.4,0.2,setosa
4.7,3.2,1.3,0.2,setosa
4.6,3.1,1.5,0.2,setosa
5.0,3.6,1.4,0.2,setosa
5.4,3.9,1.7,0.4,setosa
4.6,3.4,1.4,0.3,setosa
5.0,3.4,1.5,0.2,setosa


In [58]:
# {61} Write WekaDemo.java
weka_java = '''import weka.core.Instances; import weka.core.converters.ConverterUtils.DataSource; import weka.classifiers.trees.J48; public class WekaDemo { public static void main(String[] args) throws Exception { String path=args.length>0?args[0]:"data/iris.arff"; DataSource src=new DataSource(path); Instances data=src.getDataSet(); data.setClassIndex(data.numAttributes()-1); J48 tree=new J48(); tree.buildClassifier(data); System.out.println(tree); } }'''
with open('java/WekaDemo.java','w') as f: f.write(weka_java)

In [59]:
# {62} Compile WekaDemo
!javac -cp java/weka.jar -d java java/WekaDemo.java

In [60]:
# A J48 tree is a machine learning algorithm used for classification. It is an
# open-source Java implementation of the standard C4.5 decision tree algorithm
# (with the 'J' standing for Java). It is most commonly used within the data
# mining software Weka {63} Run WekaDemo – should print J48 tree
!wget -q https://repo1.maven.org/maven2/nz/ac/waikato/cms/weka/thirdparty/bounce/0.18/bounce-0.18.jar -O java/bounce.jar
!wget -q https://repo1.maven.org/maven2/com/googlecode/matrix-toolkits-java/mtj/1.0.4/mtj-1.0.4.jar -O java/mtj.jar
!wget -q https://repo1.maven.org/maven2/com/github/fommil/netlib/core/1.1.2/core-1.1.2.jar -O java/core.jar
!wget -q https://repo1.maven.org/maven2/net/sourceforge/f2j/arpack_combined_all/0.1/arpack_combined_all-0.1.jar -O java/arpack.jar
!echo -e "handlers=java.util.logging.ConsoleHandler\n.level=SEVERE\njava.util.logging.ConsoleHandler.level=SEVERE" > java/logging.properties
!java -Djava.util.logging.config.file=java/logging.properties -cp java:java/weka.jar:java/bounce.jar:java/mtj.jar:java/core.jar:java/arpack.jar WekaDemo data/iris.arff

J48 pruned tree
------------------

pet_wid <= 0.6: setosa (50.0)
pet_wid > 0.6
|   pet_wid <= 1.7
|   |   pet_len <= 4.9: versicolor (48.0/1.0)
|   |   pet_len > 4.9
|   |   |   pet_wid <= 1.5: virginica (3.0)
|   |   |   pet_wid > 1.5: versicolor (3.0/1.0)
|   pet_wid > 1.7: virginica (46.0/1.0)

Number of Leaves  : 	5

Size of the tree : 	9



In [61]:
# {64} Write Java API client
client_java = r'''import java.net.*; import java.net.http.*; public class ApiClient { public static void main(String[] args) throws Exception { String json=args.length>0?args[0]:"{\"features\":[5.1,3.5,1.4,0.2]}"; HttpClient c=HttpClient.newHttpClient(); HttpRequest r=HttpRequest.newBuilder().uri(URI.create("http://127.0.0.1:5000/predict")).header("Content-Type","application/json").POST(HttpRequest.BodyPublishers.ofString(json)).build(); System.out.println(c.send(r,HttpResponse.BodyHandlers.ofString()).body()); } }'''
with open('java/ApiClient.java','w') as f: f.write(client_java)

In [62]:
# {65} Compile ApiClient
!javac -d java java/ApiClient.java

In [63]:
# {66} Run Java client against Flask
import subprocess, time, requests
try:
    requests.get('http://127.0.0.1:5000/')
except requests.exceptions.ConnectionError:
    print('Restarting Flask...')
    subprocess.Popen(['python', 'app.py'])
    time.sleep(3)

!java -cp java ApiClient

{"class":"setosa","pred":0}



In [64]:
# {67} Run with different features
!java -cp java ApiClient "{\"features\":[6.0,2.9,4.5,1.5]}"

{"class":"versicolor","pred":1}



In [65]:
# {68} Verify both calls logged
pd.read_sql('SELECT id,request,response FROM api_logs ORDER BY id DESC LIMIT 5', sqlite3.connect('ml.db'))

,id,request,response
0,3,"{""features"": [6.0, 2.9, 4.5, 1.5]}","{""pred"": 1}"
1,2,"{""features"": [5.1, 3.5, 1.4, 0.2]}","{""pred"": 0}"
2,1,"{""features"": [5.1, 3.5, 1.4, 0.2]}","{""pred"": 0}"


In [66]:
# {69} Query predictions summary
pd.read_sql('SELECT pred, COUNT(*) as cnt FROM predictions GROUP BY pred', sqlite3.connect('ml.db'))

,pred,cnt
0,0,2
1,1,1


In [67]:
# {70} Join iris with predictions (demo)
pd.read_sql('SELECT i.class, p.pred FROM iris i JOIN predictions p ON p.id = i.id LIMIT 5', sqlite3.connect('ml.db'))

,class,pred
0,setosa,0
1,setosa,0
2,setosa,1


In [68]:
# {71} Load houses into pandas
conn = sqlite3.connect('ml.db')
houses_df = pd.read_sql('SELECT * FROM houses', conn)

In [69]:
# {72} Prepare tensors for GPU linear regression
Xh = torch.tensor(houses_df[['sqft']].values, dtype=torch.float32, device=device)
yh = torch.tensor(houses_df[['price']].values, dtype=torch.float32, device=device)

In [70]:
# {73} Simple linear model on GPU
lin = nn.Linear(1,1).to(device)
opt = torch.optim.SGD(lin.parameters(), lr=1e-7)

In [71]:
# {74} Train 500 steps
for i in range(500):
    opt.zero_grad()
    loss = ((lin(Xh)-yh)**2).mean()
    loss.backward()
    opt.step()

In [72]:
# {75} Show learned weight
print('weight:', lin.weight.item(), 'bias:', lin.bias.item())

weight: 197.93980407714844 bias: 0.979532778263092


In [73]:
# {76} Predict price for 1600 sqft
with torch.no_grad():
    pred_price = lin(torch.tensor([[1600.0]], device=device)).item()
print('Predicted:', int(pred_price))

Predicted: 316704


In [74]:
# {77} Log to metrics
conn.execute('INSERT INTO metrics VALUES (?,?)', ('house_gpu', float(pred_price)))
conn.commit()

In [75]:
# {78} Show all metrics
pd.read_sql('SELECT * FROM metrics', conn)

,run,acc
0,torch_gpu,0.953333
1,house_gpu,316704.656250


In [76]:
# {79} Close DB
conn.close()

In [77]:
# {80} Cleanup GPU
torch.cuda.empty_cache()

In [78]:
# {81} List all generated files
!find . -type f -name '*.py' -o -name '*.db' -o -name '*.pkl' -o -name '*.csv' | sort

./app.py
./data/iris.csv
./ml.db
./models/iris_pipe.pkl
./models/scaler.pkl
./predictions_export.csv


In [79]:
# {82} Export SQLite to CSV for download
import pandas as pd
import sqlite3

conn = sqlite3.connect('ml.db')
df_export = pd.read_sql('SELECT * FROM predictions;', conn)
df_export.to_csv('predictions_export.csv', index=False)
conn.close()

In [80]:
# {83} Show export and download to local computer
import pandas as pd
from google.colab import files

df_csv = pd.read_csv('predictions_export.csv')
display(df_csv) # Display the full dataframe instead of just .head()

# Trigger file download
files.download('predictions_export.csv')

,id,features,pred,ts
0,1,"[5.1, 3.5, 1.4, 0.2]",0,2026-05-17T23:20:27.055238
1,2,"[5.1, 3.5, 1.4, 0.2]",0,2026-05-17T23:20:30.034282
2,3,"[6.0, 2.9, 4.5, 1.5]",1,2026-05-17T23:20:30.747076


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [81]:
# {84} Stop Flask (kill python)
!pkill -f app.py

In [82]:
# {85} Final message
print('Colab Big Sprint complete - GPU trained, Flask logged, Java called, all in 85 cells!')

Colab Big Sprint complete - GPU trained, Flask logged, Java called, all in 85 cells!
